# Sciplex3 Preprocessing (Condot-Aligned)

将你的 Sciplex 数据处理为 Biolord 可训练格式，并优先复用 condot 划分列 `split_ood_finetuning`。

In [1]:
import sys
from pathlib import Path

# Clear stale module cache to avoid using old condot_compat code in a long-lived kernel
for _m in list(sys.modules):
    if _m == 'condot_compat' or _m.startswith('condot_compat.'):
        del sys.modules[_m]
    if _m == 'chemprop' or _m.startswith('chemprop.'):
        del sys.modules[_m]

sys.path.append('../../../')

# Dependency sanity check (non-invasive)
import importlib.util
import setuptools
print('setuptools version:', setuptools.__version__)
print('chemprop installed:', importlib.util.find_spec('chemprop') is not None)
print('pkg_resources available:', importlib.util.find_spec('pkg_resources') is not None)

from condot_compat import (
    load_adata,
    save_adata,
    compute_rdkit2d_features,
    apply_condot_aligned_split,
    summarize_split,
    save_config_snapshot,
)

setuptools version: 82.0.1
chemprop installed: True
pkg_resources available: True


In [2]:
# ====== 可改参数 ======
RAW_DATA_PATH = '/data/jamin_datasets/chemCPA/sciplex_complete_lincs_genes_v3.h5ad'
OUTPUT_DIR = Path('../../../results/biolord_condot_aligned/preprocessed')
OUTPUT_DATA_PATH = OUTPUT_DIR / 'sciplex3_biolord_condot_aligned.h5ad'
SPLIT_KEY_OUT = 'split_eval'
SEED = 42
VARIANCE_THRESHOLD = 0.001

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
adata = load_adata(RAW_DATA_PATH)
print('Loaded:', adata.shape)

adata = compute_rdkit2d_features(
    adata,
    smiles_col='SMILES',
    dose_col='dose',
    variance_threshold=VARIANCE_THRESHOLD,
)

used_split_key = apply_condot_aligned_split(
    adata,
    output_split_key=SPLIT_KEY_OUT,
    preferred_split_key='split_ood_finetuning',
    fallback_split_key='split_ood',
    seed=SEED,
)
print('Using split key:', used_split_key)

Loaded: (581777, 977)


/data/jamin/conda_envs/biolord/lib/python3.10/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using split key: split_eval


In [4]:
summary = summarize_split(adata, split_key=SPLIT_KEY_OUT)
summary['split_counts'].to_csv(OUTPUT_DIR / 'split_counts.csv')
summary['by_condition'].to_csv(OUTPUT_DIR / 'split_by_condition.csv', index=False)
summary['by_cell_type'].to_csv(OUTPUT_DIR / 'split_by_cell_type.csv', index=False)

save_adata(adata, str(OUTPUT_DATA_PATH))

save_config_snapshot(
    {
        'raw_data_path': RAW_DATA_PATH,
        'output_data_path': str(OUTPUT_DATA_PATH),
        'split_key_out': SPLIT_KEY_OUT,
        'seed': SEED,
        'variance_threshold': VARIANCE_THRESHOLD,
    },
    str(OUTPUT_DIR / 'preprocess_config.json')
)

print('Saved processed dataset to:', OUTPUT_DATA_PATH)
summary['split_counts']

Saved processed dataset to: ../../../results/biolord_condot_aligned/preprocessed/sciplex3_biolord_condot_aligned.h5ad


,count
split_eval,
train,538211
test,23074
ood,20492
